# PySpark Demo: RDD vs Pandas DataFrame vs Spark DataFrame

**Dataset:** NYC Yellow Taxi Trip Records — Jan 2022 + Jan 2023 (~6M rows combined)

This notebook compares three APIs across two dimensions:
- **Functionality**: API style, expressiveness, schema handling
- **Efficiency**: Memory usage, wall-clock time for common operations

| API | Abstraction | Lazy? | Distributed? | Typed schema? |
|-----|-------------|-------|--------------|---------------|
| RDD | Low-level (Python objects) | Yes | Yes | No |
| Pandas DataFrame | High-level | No (eager) | No (single node) | Yes (dtypes) |
| Spark DataFrame | High-level | Yes | Yes | Yes (StructType) |


## 0. Setup

In [1]:
# Install dependencies if needed
# !pip install pyspark pandas pyarrow matplotlib requests

In [ ]:
import os
import math
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType

print("Libraries imported successfully.")

In [ ]:
spark = (
    SparkSession.builder
    .appName("PySpark_Demo_NYC_Taxi")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# FileStreamSink.hasMetadata() always warns on glob paths in Spark 4.x — silence it
log4j = spark.sparkContext._jvm.org.apache.log4j
log4j.Logger.getLogger(
    "org.apache.spark.sql.execution.streaming.sinks.FileStreamSink"
).setLevel(log4j.Level.ERROR)

sc = spark.sparkContext

print(f"Spark version      : {spark.version}")
print(f"Master             : {spark.sparkContext.master}")
print(f"Parallel workers   : {sc.defaultParallelism}")


---
## 1. Download Both Years of NYC Taxi Data

We download **Jan 2022** and **Jan 2023** parquet files (~50 MB each, ~3M rows each).
Combining them gives us ~6M rows — enough to start seeing Spark's multi-file advantages.

In [ ]:
DATA_DIR = os.path.abspath(os.getcwd())  # absolute path suppresses FileStreamSink WARN

FILES = {
    "yellow_tripdata_2021-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet",
    "yellow_tripdata_2022-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-01.parquet",
    "yellow_tripdata_2023-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet",
    "yellow_tripdata_2024-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "yellow_tripdata_2025-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet",
}

for fname, url in FILES.items():
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f"Downloading {fname} ...")
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(fpath, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"  Saved ({os.path.getsize(fpath)/1e6:.1f} MB)")
    else:
        print(f"  Found cached: {fname} ({os.path.getsize(fpath)/1e6:.1f} MB)")

DATA_2021 = os.path.join(DATA_DIR, "yellow_tripdata_2021-01.parquet")
DATA_2022 = os.path.join(DATA_DIR, "yellow_tripdata_2022-01.parquet")
DATA_2023 = os.path.join(DATA_DIR, "yellow_tripdata_2023-01.parquet")
DATA_2024 = os.path.join(DATA_DIR, "yellow_tripdata_2024-01.parquet")
DATA_2025 = os.path.join(DATA_DIR, "yellow_tripdata_2025-01.parquet")
DATA_GLOB = os.path.join(DATA_DIR, "yellow_tripdata_202*.parquet")  # absolute glob


---
## 2. Combining Files — Three APIs, Three Styles

This is where the APIs diverge significantly in **ergonomics**.

| API | How to combine files | Parallel? |
|-----|---------------------|-----------|
| Pandas | `pd.concat([read(f1), read(f2)])` — sequential reads | No |
| Spark DF | `spark.read.parquet("*.parquet")` — wildcard, reads all files in parallel | Yes |
| RDD | Created from the Spark DF — inherits parallelism | Yes |

In [ ]:
# ── PANDAS: explicit sequential concat ──────────────────────────────────────
print("[Pandas] Reading and combining two parquet files...")
t0 = time.perf_counter()
pdf_2021 = pd.read_parquet(DATA_2021)
pdf_2022 = pd.read_parquet(DATA_2022)
pdf_2023 = pd.read_parquet(DATA_2023)
pdf_2024 = pd.read_parquet(DATA_2024)
pdf_2025 = pd.read_parquet(DATA_2025)
pdf = pd.concat([pdf_2021, pdf_2022, pdf_2023, pdf_2024, pdf_2025], ignore_index=True)
t_pandas_combine = time.perf_counter() - t0

print(f"  2021 rows : {len(pdf_2021):,}")
print(f"  2022 rows : {len(pdf_2022):,}")
print(f"  2023 rows : {len(pdf_2023):,}")
print(f"  2024 rows : {len(pdf_2024):,}")
print(f"  2025 rows : {len(pdf_2025):,}")
print(f"  Combined  : {len(pdf):,} rows")
print(f"  Time      : {t_pandas_combine:.2f}s  (sequential: read f1 → read f2 → concat)")
print(f"  RAM       : {pdf.memory_usage(deep=True).sum()/1e6:.0f} MB")

In [ ]:
# ── SPARK DF: read files in parallel, handling schema drift across years ────
# Note: 2024+ files store passenger_count as INT64 vs DoubleType in 2021-2023.
# Reading each file with its own schema then union avoids the vectorized-reader
# type-mismatch error (PARQUET_COLUMN_DATA_TYPE_MISMATCH).
print("[Spark DF] Reading all years with schema-safe union...")

t0 = time.perf_counter()

def _read_year(path):
    df = spark.read.parquet(path)
    # Normalize passenger_count to LongType — 2024+ changed it from DoubleType
    return df.withColumn("passenger_count", F.col("passenger_count").cast("long"))

all_files = [DATA_2021, DATA_2022, DATA_2023, DATA_2024, DATA_2025]
sdf = _read_year(all_files[0])
for _path in all_files[1:]:
    sdf = sdf.unionByName(_read_year(_path), allowMissingColumns=True)

plan_time = time.perf_counter() - t0
print(f"  Plan built in {plan_time*1000:.1f} ms  ← lazy, no I/O yet")

t0 = time.perf_counter()
sdf_count = sdf.count()
t_spark_combine = time.perf_counter() - t0

print(f"  Combined  : {sdf_count:,} rows")
print(f"  Time      : {t_spark_combine:.2f}s  (parallel: all files read simultaneously)")
print(f"  Partitions: {sdf.rdd.getNumPartitions()}")
sdf.show(5, truncate=False)

In [ ]:
# ── Schema alignment check ────────────────────────────────────────────────────
# Real-world concern: schemas can drift between years (columns added/renamed/retyped)
print("Checking for schema differences between 2022 and 2023 ...")

schema_2022 = {f.name: str(f.dataType) for f in spark.read.parquet(DATA_2022).schema.fields}
schema_2023 = {f.name: str(f.dataType) for f in spark.read.parquet(DATA_2023).schema.fields}

only_in_2022 = set(schema_2022) - set(schema_2023)
only_in_2023 = set(schema_2023) - set(schema_2022)
type_changed = {c for c in set(schema_2022) & set(schema_2023) if schema_2022[c] != schema_2023[c]}

if not any([only_in_2022, only_in_2023, type_changed]):
    print("  Schemas match perfectly — safe to concat.")
else:
    if only_in_2022:  print(f"  Only in 2022 : {only_in_2022}")
    if only_in_2023:  print(f"  Only in 2023 : {only_in_2023}")
    if type_changed:  print(f"  Type changed : {type_changed}")
    print("  Note: Spark handles schema evolution automatically with mergeSchema option.")

print(f"\nTotal columns: {len(schema_2023)}")
sdf.printSchema()

In [ ]:
# ── RDD from combined Spark DF ───────────────────────────────────────────────
rdd = sdf.rdd
print(f"RDD created (lazy). Partitions: {rdd.getNumPartitions()}")
print(f"Sample: {rdd.first()}")

### 3. Lazy vs Eager

In [ ]:
print("=" * 55)
print("PANDAS  (eager) — executes immediately")
print("=" * 55)
t0 = time.perf_counter()
_ = pdf[pdf["fare_amount"] > 10]
print(f"  pdf[...] took: {(time.perf_counter()-t0)*1000:.1f} ms")

print()
print("=" * 55)
print("SPARK DF (lazy) — plan only until action")
print("=" * 55)
t0 = time.perf_counter()
lazy_plan = sdf.filter(F.col("fare_amount") > 10)
print(f"  sdf.filter() plan: {(time.perf_counter()-t0)*1000:.2f} ms ← no I/O")
t0 = time.perf_counter()
_ = lazy_plan.count()
print(f"  .count() action  : {time.perf_counter()-t0:.2f}s")

print()
print("=" * 55)
print("RDD     (lazy) — lineage only until action")
print("=" * 55)
t0 = time.perf_counter()
lazy_rdd = rdd.filter(lambda r: r.fare_amount is not None and r.fare_amount > 10)
print(f"  rdd.filter() plan: {(time.perf_counter()-t0)*1000:.2f} ms ← no I/O")
t0 = time.perf_counter()
_ = lazy_rdd.count()
print(f"  .count() action  : {time.perf_counter()-t0:.2f}s")

---
## 4. DataFrame API: Deep Dive

The DataFrame API is the primary interface for data engineering in PySpark. This section covers reading from multiple formats, transformations, joins, window functions, UDFs, and null handling.

| Topic | What you will learn |
|-------|-------------------|
| Reading data | CSV / JSON / Parquet with schema inference vs explicit schemas |
| Column ops | `withColumn`, `cast`, `rename`, `drop` |
| Filtering | expression / SQL string / `between` / `isin` / null-safe |
| Aggregations | `count`, `sum`, `avg`, `stddev`, `percentile_approx`, `groupBy` |
| Joins | inner / left / anti + broadcast hint |
| Window functions | `rank`, `lag`, `lead`, `ntile`, `cumsum` |
| Null handling | `dropna`, `fillna`, `coalesce`, forward-fill |
| UDFs | regular UDF vs Pandas UDF (vectorized) — performance tradeoff |
| Built-in functions | date/time, math, string, conditional, struct/array |

In [ ]:
import shutil

# Write one year of taxi data to CSV and JSON using Spark (no pandas intermediary)
# Use coalesce(1) so each format lands as a single file for easy reading demos
CSV_DIR  = os.path.join(DATA_DIR, "taxi_csv")
JSON_DIR = os.path.join(DATA_DIR, "taxi_json")

sdf_2023_sample = (
    sdf.filter(F.year("tpep_pickup_datetime") == 2023)
       .select("VendorID","tpep_pickup_datetime","trip_distance",
               "fare_amount","payment_type","PULocationID","DOLocationID")
       .limit(50_000)
)

for path in [CSV_DIR, JSON_DIR]:
    if os.path.exists(path): shutil.rmtree(path)

sdf_2023_sample.coalesce(1).write.option("header", True).mode("overwrite").csv(CSV_DIR)
sdf_2023_sample.coalesce(1).write.mode("overwrite").json(JSON_DIR)

# Resolve the single part file inside each directory
CSV_PATH  = next(os.path.join(CSV_DIR, f)  for f in os.listdir(CSV_DIR)  if f.endswith(".csv"))
JSON_PATH = next(os.path.join(JSON_DIR, f) for f in os.listdir(JSON_DIR) if f.endswith(".json"))

print(f"CSV : {os.path.getsize(CSV_PATH)/1e6:.1f} MB  → {CSV_PATH}")
print(f"JSON: {os.path.getsize(JSON_PATH)/1e6:.1f} MB  → {JSON_PATH}")

In [ ]:
# --- Reading formats: schema inference vs explicit schema ---
from pyspark.sql.types import StructType, StructField, LongType, TimestampType, DoubleType

# 1. Parquet — schema embedded, always use this in production
print("=== Parquet (schema embedded, no inference cost) ===")
sdf_parquet = spark.read.parquet(DATA_2023)
sdf_parquet.printSchema()
print(f"Rows: {sdf_parquet.count():,}")

# 2. CSV — schema inference scans file to guess types (slow on large files)
print("\n=== CSV: schema inference ===")
sdf_csv_infer = spark.read.option("header", True).option("inferSchema", True).csv(CSV_PATH)
sdf_csv_infer.printSchema()

# Explicit schema — preferred in production: no scan, type-safe
taxi_csv_schema = StructType([
    StructField("VendorID",             LongType(),      True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("trip_distance",        DoubleType(),    True),
    StructField("fare_amount",          DoubleType(),    True),
    StructField("payment_type",         LongType(),      True),
    StructField("PULocationID",         LongType(),      True),
    StructField("DOLocationID",         LongType(),      True),
])
print("=== CSV: explicit schema (fast, type-safe) ===")
sdf_csv_explicit = spark.read.option("header", True).schema(taxi_csv_schema).csv(CSV_PATH)
sdf_csv_explicit.printSchema()
print(f"Rows: {sdf_csv_explicit.count():,}")

# 3. JSON — schema inferred from a sample of the file
print("=== JSON (schema inferred) ===")
sdf_json = spark.read.option("multiline", False).json(JSON_PATH)
sdf_json.printSchema()
sdf_json.show(3)

print("\nAll subsequent operations use sdf — the full 13M-row taxi dataset.")

In [ ]:
# --- Column operations ---
sdf_cols = (
    sdf
    .withColumn("fare_usd",    F.round(F.col("fare_amount"), 2))
    .withColumn("tip_pct",     F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 1))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumnRenamed("PULocationID", "pickup_zone")
    .drop("extra", "mta_tax", "improvement_surcharge")
)
print("New columns: fare_usd, tip_pct, pickup_hour, pickup_date")
sdf_cols.select("fare_usd", "tip_pct", "pickup_hour", "pickup_date", "pickup_zone").show(5)

# --- Filtering techniques ---
print("=== Filtering Techniques ===")
# Column expression
f1 = sdf.filter(F.col("fare_amount") > 50)
print(f"fare > $50                          : {f1.count():,} rows")

# SQL string
f2 = sdf.filter("fare_amount > 50 AND trip_distance > 10")
print(f"fare > $50 AND dist > 10mi          : {f2.count():,} rows")

# between + isin
f3 = sdf.filter(F.col("passenger_count").between(1, 4) & F.col("payment_type").isin([1, 2]))
print(f"1-4 passengers, card/cash           : {f3.count():,} rows")

# NOT NULL
f4 = sdf.filter(F.col("passenger_count").isNotNull() & (F.col("fare_amount") > 0))
print(f"non-null passengers, positive fare  : {f4.count():,} rows")

In [ ]:
# --- Aggregations ---
print("=== Aggregation Functions ===")
agg_result = sdf.filter(F.col("fare_amount") > 0).agg(
    F.count("*").alias("total_trips"),
    F.round(F.sum("fare_amount"), 0).alias("total_revenue"),
    F.round(F.avg("fare_amount"), 2).alias("mean_fare"),
    F.round(F.stddev("fare_amount"), 2).alias("stddev_fare"),
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.percentile_approx("fare_amount", 0.5).alias("median_fare"),
    F.percentile_approx("fare_amount", [0.25, 0.75]).alias("iqr_fares"),
    F.countDistinct("PULocationID").alias("unique_pickup_zones"),
)
agg_result.show(1, truncate=False)

# GroupBy: hourly patterns
print("=== GroupBy: Hourly trip patterns ===")
hourly = (
    sdf
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .groupBy("hour")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_dist_mi"),
        F.round(F.sum("tip_amount") / F.sum("fare_amount") * 100, 1).alias("tip_rate_pct"),
    )
    .orderBy("hour")
)
hourly.show(24)

In [ ]:
# --- Joins ---
# Small zone lookup table
zone_data = [
    (132, "JFK Airport"),         (138, "LaGuardia Airport"),
    (161, "Midtown Center"),      (236, "Upper East Side N"),
    (237, "Upper East Side S"),   (142, "Lincoln Square E"),
    (230, "Times Sq/Theatre"),    (170, "Murray Hill"),
]
zone_df = spark.createDataFrame(zone_data, ["LocationID", "Zone"])

# Inner join: trips from known zones only
print("=== Inner Join ===")
inner = (
    sdf.alias("t")
    .join(zone_df.alias("z"), F.col("t.PULocationID") == F.col("z.LocationID"), "inner")
    .select("tpep_pickup_datetime", "Zone", "fare_amount", "trip_distance")
)
print(f"Trips from named zones: {inner.count():,}")
inner.show(5)

# Left join: all trips, zone name if available
print("=== Left Join ===")
left = (
    sdf.alias("t")
    .join(zone_df.alias("z"), F.col("t.PULocationID") == F.col("z.LocationID"), "left")
    .select(F.coalesce(F.col("Zone"), F.lit("Other")).alias("Zone"), "fare_amount")
)
left.groupBy("Zone").agg(F.count("*").alias("trips"), F.round(F.avg("fare_amount"),2).alias("avg_fare")).orderBy(F.col("trips").desc()).show(10)

# Anti join: trips NOT from known zones
print("=== Anti Join (left_anti) ===")
anti = sdf.join(zone_df, sdf["PULocationID"] == zone_df["LocationID"], "left_anti")
print(f"Trips from zones not in lookup: {anti.count():,}")

In [ ]:
# --- Window Functions ---
from pyspark.sql.window import Window

print("=== Window Functions ===")

# Partition by zone, order by fare descending
w_zone = Window.partitionBy("PULocationID").orderBy(F.col("fare_amount").desc())
# Running cumulative sum within each zone, ordered by pickup time
w_running = Window.partitionBy("PULocationID").orderBy("tpep_pickup_datetime").rowsBetween(Window.unboundedPreceding, 0)

sdf_win = (
    sdf.filter(F.year("tpep_pickup_datetime") == 2023)
       .filter(F.col("fare_amount") > 0)
       .withColumn("rank_in_zone",  F.rank().over(w_zone))
       .withColumn("dense_rank",    F.dense_rank().over(w_zone))
       .withColumn("lag_fare",      F.lag("fare_amount", 1).over(w_zone))
       .withColumn("lead_fare",     F.lead("fare_amount", 1).over(w_zone))
       .withColumn("pct_rank",      F.round(F.percent_rank().over(w_zone), 3))
       .withColumn("cum_revenue",   F.sum("fare_amount").over(w_running))
)

print("Top 3 most expensive trips per pickup zone:")
sdf_win.filter(F.col("rank_in_zone") <= 3).select(
    "PULocationID","fare_amount","rank_in_zone","dense_rank","lag_fare","pct_rank"
).orderBy("PULocationID","rank_in_zone").show(15)

# ntile: divide trips into 4 fare quartiles within each zone
w_ntile = Window.partitionBy("PULocationID").orderBy("fare_amount")
sdf_ntile = (
    sdf.filter(F.year("tpep_pickup_datetime") == 2023).filter(F.col("fare_amount") > 0)
       .withColumn("quartile", F.ntile(4).over(w_ntile))
       .withColumn("row_num",  F.row_number().over(w_ntile))
)
print("Quartile distribution across all zones:")
sdf_ntile.groupBy("quartile").agg(F.count("*").alias("trips"), F.round(F.avg("fare_amount"),2).alias("avg_fare")).orderBy("quartile").show()

In [ ]:
# --- Handling Nulls ---
print("=== Null Handling ===")

# Count nulls per column
total_rows = sdf.count()
null_counts = sdf.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf.columns
]).toPandas().T.rename(columns={0: "null_count"})
null_counts["null_pct"] = (null_counts["null_count"] / total_rows * 100).round(2)
print("Columns with nulls:")
print(null_counts[null_counts["null_count"] > 0].sort_values("null_count", ascending=False))

print("\n--- Strategy 1: dropna() — drop rows with ANY null ---")
sdf_no_nulls = sdf.dropna()
print(f"  {sdf_no_nulls.count():,} rows remain (from {total_rows:,})")

print("\n--- Strategy 2: fillna() — replace nulls with defaults ---")
sdf_filled = sdf.fillna({"passenger_count": 1, "airport_fee": 0.0, "congestion_surcharge": 0.0, "store_and_fwd_flag": "N"})
sdf_filled.select("passenger_count","airport_fee","congestion_surcharge").show(3)

print("\n--- Strategy 3: coalesce() — pick first non-null from candidates ---")
sdf_coalesce = sdf.withColumn("fee", F.coalesce(F.col("airport_fee"), F.col("congestion_surcharge"), F.lit(0.0)))
sdf_coalesce.select("airport_fee","congestion_surcharge","fee").show(5)

print("\n--- Strategy 4: forward-fill with last() over window ---")
w_ff = Window.orderBy("tpep_pickup_datetime").rowsBetween(Window.unboundedPreceding, 0)
sdf_ffill = sdf.withColumn(
    "pax_filled", F.last("passenger_count", ignorenulls=True).over(w_ff)
)
sdf_ffill.select("tpep_pickup_datetime","passenger_count","pax_filled").filter(F.col("passenger_count").isNull()).show(5)

In [ ]:
# --- UDFs: regular vs Pandas UDF (vectorized) ---
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType, DoubleType as DT
import pandas as pd

print("=== UDFs — Performance Warning ===")
print("Regular UDFs: row-by-row Python deserialization → slow")
print("Pandas UDFs : batch-oriented via Apache Arrow → much faster\n")

# Regular UDF (use only when no built-in equivalent exists)
def fare_band_fn(fare):
    if fare is None: return "unknown"
    if fare < 5:    return "micro"
    if fare < 15:   return "short"
    if fare < 30:   return "medium"
    return "long"

fare_band_udf = udf(fare_band_fn, StringType())

# Pandas UDF (vectorized — operates on pd.Series, uses Arrow)
@pandas_udf(DT())
def tip_ratio_pudf(fare: pd.Series, tip: pd.Series) -> pd.Series:
    return (tip / fare.replace(0, float("nan"))).fillna(0.0)

sdf_sample_udf = sdf.filter(F.col("fare_amount") > 0).limit(200_000)

t0 = time.perf_counter()
r1 = sdf_sample_udf.withColumn("band", fare_band_udf("fare_amount")).groupBy("band").count().collect()
t_udf = time.perf_counter() - t0

t0 = time.perf_counter()
r2 = sdf_sample_udf.withColumn("tip_ratio", tip_ratio_pudf("fare_amount","tip_amount")).agg(F.avg("tip_ratio")).collect()
t_pudf = time.perf_counter() - t0

print(f"Regular UDF  (200K rows): {t_udf:.2f}s")
print(f"Pandas UDF   (200K rows): {t_pudf:.2f}s")
print("\nFare bands:")
for row in sorted(r1, key=lambda r: r["count"], reverse=True):
    print(f"  {row['band']:8s}: {row['count']:,}")

print("""
Rule: prefer built-in functions — they run in the JVM and bypass Python entirely.
Built-in equivalent of fare_band_fn:
  F.when(F.col("fare_amount") < 5, "micro")
   .when(F.col("fare_amount") < 15, "short")
   .when(F.col("fare_amount") < 30, "medium")
   .otherwise("long")
""")

In [ ]:
# --- Built-in functions showcase ---
print("=== Built-in Functions (JVM-native, no Python overhead) ===")

sdf_builtins = (
    sdf.filter(F.col("fare_amount") > 0)
    # String
    .withColumn("flag_upper",   F.upper("store_and_fwd_flag"))
    # Date/time
    .withColumn("pickup_date",  F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour",  F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow",   F.dayofweek("tpep_pickup_datetime"))  # 1=Sun…7=Sat
    .withColumn("duration_min", F.round(
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60, 1
    ))
    # Math
    .withColumn("fare_log",  F.log("fare_amount"))
    .withColumn("fare_ceil", F.ceil("fare_amount"))
    # Conditional
    .withColumn("is_airport",  F.col("RatecodeID").isin([2, 3]).cast("boolean"))
    .withColumn("surge_flag",  F.when(F.col("fare_amount") > 50, "surge").otherwise("normal"))
    .withColumn("fare_band",
        F.when(F.col("fare_amount") < 5,  "micro")
         .when(F.col("fare_amount") < 15, "short")
         .when(F.col("fare_amount") < 30, "medium")
         .otherwise("long")
    )
    # Hash: anonymize sensitive fields
    .withColumn("vendor_hash", F.md5(F.col("VendorID").cast("string")))
)
sdf_builtins.select("pickup_hour","pickup_dow","duration_min","fare_log","is_airport","surge_flag","fare_band").show(5)

# Array / struct functions (useful for nested/semi-structured data)
print("=== collect_list, collect_set, struct ===")
(
    sdf.filter(F.year("tpep_pickup_datetime") == 2023)
    .groupBy(F.hour("tpep_pickup_datetime").alias("hour"))
    .agg(
        F.collect_set("payment_type").alias("payment_types_used"),
        F.struct(F.min("fare_amount").alias("min"), F.max("fare_amount").alias("max")).alias("fare_range"),
    )
    .orderBy("hour")
    .show(5, truncate=False)
)

---
## 13. Section 7 — Spark SQL

Spark SQL lets you query DataFrames using standard SQL. Both APIs compile to the same Catalyst query plan — use whichever is clearer.

| Concept | Description |
|---------|-------------|
| `createOrReplaceTempView` | Session-scoped view — gone when SparkSession ends |
| `createGlobalTempView` | Cross-session view in `global_temp` database |
| CTE (`WITH`) | Readable multi-step queries |
| `EXPLAIN` | Inspect the physical plan Spark will execute |
| Predicate pushdown | Spark passes filters to the storage layer (Parquet row groups) |

In [ ]:
# Register temp views
sdf.createOrReplaceTempView("taxi")
zone_df.createOrReplaceTempView("zones")

# List views in the session catalog
print("Session views:", [t.name for t in spark.catalog.listTables() if t.tableType == "TEMPORARY"])

# CTE query: top revenue hours per year
result = spark.sql("""
    WITH hourly AS (
        SELECT
            YEAR(tpep_pickup_datetime)                              AS yr,
            HOUR(tpep_pickup_datetime)                              AS hr,
            COUNT(*)                                                AS trips,
            ROUND(AVG(fare_amount), 2)                              AS avg_fare,
            ROUND(SUM(tip_amount) / NULLIF(SUM(fare_amount),0)*100, 1) AS tip_pct,
            ROUND(SUM(fare_amount), 0)                              AS revenue
        FROM taxi
        WHERE fare_amount > 0
          AND YEAR(tpep_pickup_datetime) IN (2022, 2023, 2024, 2025)
        GROUP BY yr, hr
    ),
    ranked AS (
        SELECT *, RANK() OVER (PARTITION BY yr ORDER BY revenue DESC) AS rev_rank
        FROM hourly
    )
    SELECT yr, hr, trips, avg_fare, tip_pct, revenue, rev_rank
    FROM ranked
    WHERE rev_rank <= 3
    ORDER BY yr, rev_rank
""")
print("\nTop-3 revenue hours per year:")
result.show(20)

In [ ]:
# --- SQL vs DataFrame API: identical logical plan ---
print("=== SQL vs DataFrame API: same Catalyst plan ===\n")

t0 = time.perf_counter()
sql_r = spark.sql("""
    SELECT payment_type, COUNT(*) AS trips,
           ROUND(AVG(fare_amount),2) AS avg_fare,
           ROUND(SUM(tip_amount),0)  AS total_tips
    FROM taxi
    WHERE fare_amount > 0 AND YEAR(tpep_pickup_datetime) = 2023
    GROUP BY payment_type ORDER BY trips DESC
""").collect()
t_sql = time.perf_counter() - t0

t0 = time.perf_counter()
df_r = (
    sdf.filter((F.col("fare_amount") > 0) & (F.year("tpep_pickup_datetime") == 2023))
    .groupBy("payment_type")
    .agg(F.count("*").alias("trips"), F.round(F.avg("fare_amount"),2).alias("avg_fare"),
         F.round(F.sum("tip_amount"),0).alias("total_tips"))
    .orderBy(F.col("trips").desc())
).collect()
t_df = time.perf_counter() - t0

print(f"SQL time           : {t_sql:.3f}s")
print(f"DataFrame API time : {t_df:.3f}s")
print(f"Results match      : {sorted([r['trips'] for r in sql_r]) == sorted([r['trips'] for r in df_r])}")
print("\nBoth routes produce the SAME physical plan (Catalyst optimizer is API-agnostic).")

# --- Explain plan: see what Spark will actually do ---
print("\n=== Query Explain Plan ===")
query_plan = (
    sdf.filter(F.col("fare_amount") > 20)
    .groupBy(F.year("tpep_pickup_datetime").alias("year"))
    .agg(F.sum("fare_amount").alias("revenue"))
    .orderBy("year")
)
query_plan.explain(mode="simple")

print("\n--- Predicate pushdown: Spark tells Parquet to skip row groups ---")
sdf_pred = spark.read.parquet(DATA_2023).filter(F.col("fare_amount") > 50)
sdf_pred.explain(mode="simple")
print("Look for 'PushedFilters' in the plan — that confirms pushdown is active.")